# 05. [Retrieval 1] OpenAI 임베딩 모델·Chroma 설정 확정
  임베딩 모델 설정 확정 부분

In [37]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

In [38]:
import time # 속도 측정
from langchain_openai import OpenAIEmbeddings # 임베딩 모델

# 임베딩 모델 생성
models = {
    "OpenAI-Small": OpenAIEmbeddings(model="text-embedding-3-small"),
    "OpenAI-Large": OpenAIEmbeddings(model="text-embedding-3-large")
}

In [39]:
import pandas as pd
import numpy as np
import json
from sklearn.metrics.pairwise import cosine_similarity

# 1. 가상 데이터 로드
chunks = []
with open('../samples/processed/sample_chunks.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        chunks.append(json.loads(line))

texts = [chunk["text"] for chunk in chunks]
query = "공공 AI 학습지원 플랫폼의 총사업예산과 수행 기간은?" # 테스트 질문

# 테스트 로직
results = []
for name, model in models.items():
    # 임베딩 생성 및 시간 측정
    start_time = time.time()
    chunk_embeddings = model.embed_documents(texts)
    query_embedding = model.embed_query(query)
    duration = time.time() - start_time
    
    # 유사도 계산 (가장 높은 점수 확인)
    similarities = cosine_similarity([query_embedding], chunk_embeddings)[0]
    best_idx = np.argmax(similarities)
    
    results.append({
        "Model": name,
        "Time(sec)": round(duration, 4), # 소요시간, 0.0001초단위로 반올림
        "Best_Score": round(similarities[best_idx], 4), # 유사도 점수, 0.0001단위로 반올림
        "Best_Chunk": f"Chunk_{best_idx+1}" # 가장 유사한 Chunk ID
    })


In [40]:
df_results = pd.DataFrame(results)
print(df_results)

          Model  Time(sec)  Best_Score Best_Chunk
0  OpenAI-Small     1.8004      0.5806    Chunk_1
1  OpenAI-Large     0.8352      0.5267    Chunk_1


In [47]:
# Small 모델 토큰 / 비용
from tiktoken import encoding_for_model

enc = encoding_for_model("text-embedding-3-small")
tokens = sum(len(enc.encode(t)) for t in texts)
cost = tokens / 1000 * 0.00002  # small 모델 가격 (구글링))
print(f" Small 모델 토큰 수: {tokens}, 비용: {cost}")

 Small 모델 토큰 수: 1629, 비용: 3.258e-05


In [ ]:
# large 모델 토큰 / 비용
enc = encoding_for_model("text-embedding-3-large")
tokens = sum(len(enc.encode(t)) for t in texts)
cost = tokens / 1000 * 0.00013  # large 모델 가격 (구글링)
print(f" Large 모델 토큰 수: {tokens}, 비용: {cost}")

 Large 모델 토큰 수: 1629, 비용: 0.00021177


In [ ]:
query = "공공 AI 학습지원 플랫폼의 총사업예산과 수행 기간은?"

# 1. 모델 설정 (Best Score가 높은 Small 모델)
model = OpenAIEmbeddings(model="text-embedding-3-small")
query_embedding = model.embed_query(query)
chunk_embeddings = model.embed_documents(texts)

# 2. 유사도 계산
similarities = cosine_similarity([query_embedding], chunk_embeddings)[0]

# 3. 가장 유사도가 높은 청크의 인덱스 찾기
best_idx = np.argmax(similarities)

# 4. 상세 내용 출력
print(f"- 청크 ID: {chunks[best_idx]['chunk_id']}")
print(f"- 유사도 점수: {round(similarities[best_idx], 4)}")
print(f"- 본문 내용:\n{chunks[best_idx]['text']}")

- 청크 ID: sample_rfp_chunk_0001
- 유사도 점수: 0.5806
- 본문 내용:
[가상 제안요청서] 이 문서는 RAG 시스템 개발과 검색 성능 실험을 위해 작성한 가상 데이터입니다. 실제 기관, 사업 또는 입찰 공고와 관련이 없습니다. 1. 사업 개요 사업명은 "2026년 공공 AI 학습지원 플랫폼 구축 사업"이며 발주기관은 가상디지털진흥원이다. 사업 목적은 공공기관 임직원이 직무별 AI 교육을 수강하고 학습 이력과 역량 진단 결과를 한 곳에서 관리할 수 있는 통합 플랫폼을 구축하는 것이다. 총사업예산은 부가가치세를 포함하여 480,000,000원이며 수행 기간은 계약 체결일로부터 8개월이다. 2. 주요 기능 요구사항 사용자는 기관 계정으로 로그인하고 직무, 숙련도, 관심 분야에 따라 교육과정을 추천받을 수 있어야 한다. 학습자는 동영상 강의, 문서형 교재, 퀴즈를 이용하고 진도율과 평가 결과를 확인할 수 있어야 한다. 관리자는 교육과정 등록, 수강 승인, 학습 이력 조회, 기관별 통계 다운로드 기능을 사용할 수 있어야 한다. 플랫폼은 질의응답을 지원하는 AI 학습 도우미를 제공하되, 답변에 사용한 교육자료의 제목과 위치를 함께 표시해야 한다. 3. 데이터 및 연계 요구사항 기존 교육시스템에서 제공하는 사용자 정보와 최근 3년간의 학습 이력을 이전해야 한다. 사용자 정보에는 사번, 소속기관, 부서, 직급이 포함되며 주민등록번호는 이전 대상에서 제외한다. 외부 인사관리시스템과는 REST API 방식으로 연계하고, 매일 오전 2시에 변경된 사용자 정보를 동기화해야 한다. 데이터 이전 결과는 전체 건수, 성공 건수, 실패 건수로 구분한 검증 보고서로 제출해야 한다. 4. 성능 및 품질 요구사항


# 06. [Retrieval 1] OpenAI + Chroma 기본 유사도 검색 구현


In [81]:
import os
import sys
import yaml

# 노트북 폴더(notebook/)에서 상위 루트 경로를 인식시키기 위함
sys.path.append("../")

from src.openai_chroma_retriever import OpenAIChromaRetriever

# 1. 실제 내장된 YAML 파일 읽기
yaml_path = "../config/default.yaml"
with open(yaml_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print("✅ YAML 설정 파일 로드 성공!")
print(f"-> active_profile: {config['retrieval']['active_profile']}")
print(f"-> 기본 검색 개수(top_k): {config['retrieval']['top_k']}") # yaml에서 수정할때마다 바뀜
print(f"-> 샘플 청크 경로: {config['paths']['chunks']}")

✅ YAML 설정 파일 로드 성공!
-> active_profile: baseline
-> 기본 검색 개수(top_k): 2
-> 샘플 청크 경로: samples/processed/sample_chunks.jsonl


In [76]:
import json

# 2. YAML에 지정된 경로를 기반으로 진짜 샘플 파일 열기
# 노트북 기준 상대 경로 처리를 위해 앞에 '../'를 붙여줍니다.
chunks_real_path = "../" + config["paths"]["chunks"]

chunks = []
with open(chunks_real_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            chunks.append(json.loads(line))

print(f"✅ 실제 샘플 청크 데이터 로드 성공! (총 {len(chunks)}개 청크)")
# 데이터가 어떻게 생겼는지 첫 번째 청크만 살짝 확인
print("\n[첫 번째 청크 샘플 미리보기]")
print(json.dumps(chunks[0], indent=2, ensure_ascii=False))

✅ 실제 샘플 청크 데이터 로드 성공! (총 3개 청크)

[첫 번째 청크 샘플 미리보기]
{
  "chunk_id": "sample_rfp_chunk_0001",
  "doc_id": "sample_rfp",
  "text": "[가상 제안요청서] 이 문서는 RAG 시스템 개발과 검색 성능 실험을 위해 작성한 가상 데이터입니다. 실제 기관, 사업 또는 입찰 공고와 관련이 없습니다. 1. 사업 개요 사업명은 \"2026년 공공 AI 학습지원 플랫폼 구축 사업\"이며 발주기관은 가상디지털진흥원이다. 사업 목적은 공공기관 임직원이 직무별 AI 교육을 수강하고 학습 이력과 역량 진단 결과를 한 곳에서 관리할 수 있는 통합 플랫폼을 구축하는 것이다. 총사업예산은 부가가치세를 포함하여 480,000,000원이며 수행 기간은 계약 체결일로부터 8개월이다. 2. 주요 기능 요구사항 사용자는 기관 계정으로 로그인하고 직무, 숙련도, 관심 분야에 따라 교육과정을 추천받을 수 있어야 한다. 학습자는 동영상 강의, 문서형 교재, 퀴즈를 이용하고 진도율과 평가 결과를 확인할 수 있어야 한다. 관리자는 교육과정 등록, 수강 승인, 학습 이력 조회, 기관별 통계 다운로드 기능을 사용할 수 있어야 한다. 플랫폼은 질의응답을 지원하는 AI 학습 도우미를 제공하되, 답변에 사용한 교육자료의 제목과 위치를 함께 표시해야 한다. 3. 데이터 및 연계 요구사항 기존 교육시스템에서 제공하는 사용자 정보와 최근 3년간의 학습 이력을 이전해야 한다. 사용자 정보에는 사번, 소속기관, 부서, 직급이 포함되며 주민등록번호는 이전 대상에서 제외한다. 외부 인사관리시스템과는 REST API 방식으로 연계하고, 매일 오전 2시에 변경된 사용자 정보를 동기화해야 한다. 데이터 이전 결과는 전체 건수, 성공 건수, 실패 건수로 구분한 검증 보고서로 제출해야 한다. 4. 성능 및 품질 요구사항",
  "metadata": {
    "file_name": "sample_rfp.txt",
    "source_pat

In [78]:
# 3. 환경변수에 API Key가 잘 들어있는지 최종 확인
if "OPENAI_API_KEY" not in os.environ:
    print("❌ 경고: OPENAI_API_KEY가 설정되어 있지 않습니다. 검색 시 에러가 발생할 수 있습니다.")

# 4. 리트리버 객체 생성 (전체 config를 그대로 전달)
retriever = OpenAIChromaRetriever(chunks=chunks, config=config)

# 5. 실제 문서 내용에 있을 법한 키워드로 테스트 질문 투척!
test_query = "사업 제안서 서류 제출 마감 기한과 필수 제출 서류 목록이 어떻게 되나요?" 
print(f"🔍 테스트 질문: '{test_query}'\n")

# 6. 검색 실행
results = retriever.search(test_query)

# 7. 출력 결과 확인
print(f"🎯 검색된 근거 청크 개수: {len(results)}개 (YAML에 설정된 top_k 만큼 나옵니다)\n")
for idx, res in enumerate(results, 1):
    print(f"[{idx}순위 근거]")
    print(f"- chunk_id: {res.chunk_id}")
    print(f"- doc_id: {res.doc_id}")
    print(f"- score (거리 값): {res.score:.4f}")
    print(f"- 본문 텍스트: {res.text}")
    print("-" * 60)

🔍 테스트 질문: '사업 제안서 서류 제출 마감 기한과 필수 제출 서류 목록이 어떻게 되나요?'

🎯 검색된 근거 청크 개수: 3개 (YAML에 설정된 top_k 만큼 나옵니다)

[1순위 근거]
- chunk_id: sample_rfp_chunk_0003
- doc_id: sample_rfp
- score (거리 값): 1.1338
- 본문 텍스트: 이해도, 수행 방안, 인력 구성, 데이터 이전 계획, 보안 대책이 포함된다. 7. 산출물 착수계획서, 요구사항정의서, 화면설계서, API 명세서, 데이터 이전 결과서, 시험 결과서, 관리자 및 사용자 매뉴얼, 보안 점검 결과서, 완료보고서를 제출해야 한다. 모든 산출물은 전자파일로 제출하고 발주기관 검토 의견을 반영한 최종본을 사업 종료 전까지 제공해야 한다.
------------------------------------------------------------
[2순위 근거]
- chunk_id: sample_rfp_chunk_0002
- doc_id: sample_rfp
- score (거리 값): 1.3111
- 본문 텍스트: REST API 방식으로 연계하고, 매일 오전 2시에 변경된 사용자 정보를 동기화해야 한다. 데이터 이전 결과는 전체 건수, 성공 건수, 실패 건수로 구분한 검증 보고서로 제출해야 한다. 4. 성능 및 품질 요구사항 동시 접속자 1,000명 환경에서 일반 화면의 평균 응답시간은 3초 이내여야 한다. 교육과정 검색 결과는 5초 이내에 표시되어야 하며 월간 서비스 가용성은 99.5% 이상이어야 한다. 제안사는 오픈 전에 부하 테스트를 수행하고 동시 접속자 수, 평균 응답시간, 오류율을 포함한 시험 결과서를 제출해야 한다. 5. 보안 요구사항 모든 통신 구간에 TLS 1.2 이상을 적용하고 관리자 계정에는 다중요소인증을 적용해야 한다. 개인정보는 저장 시 암호화해야 하며 개인정보 조회와 다운로드 기록을 1년간 보관해야 한다. AI 학습 도우미는 검색된 교육자료에 없는 내용을 사실처럼 

In [84]:
# 1. 객체 생성
retriever = OpenAIChromaRetriever(chunks=chunks, config=config)

# 2. 클래스 내부 변수 직접 출력해보기
print("--- [디버그 수치 검증] ---")
print("Retriever가 최종 인지한 default_top_k 값:", retriever.default_top_k)

# 3. 진짜 검색 결과 개수 확인
results = retriever.search("사업 제안서 서류 제출 마감 기한")
print("실제 받아온 검색 결과 개수:", len(results))

--- [디버그 수치 검증] ---
Retriever가 최종 인지한 default_top_k 값: 3
실제 받아온 검색 결과 개수: 3


In [ ]:
%load_ext autoreload
%autoreload 2

: 